# MGMT 590 — LendingClub Loan Default Risk (Indiana Borrowers)
## Phase 1: Project Architecture & Data Foundation

**Course:** MGMT 59000, Summer 2026, Section DY2 — Purdue University
**Scope of this notebook:** data ingestion, validation, cleaning, feature
engineering, leakage-safe train/validation/test splitting, and
preprocessing-pipeline construction/serialization.

This notebook is a thin, readable wrapper around the reusable project
modules `src/config.py`, `src/utils.py`, and `src/train_models.py`. All
real logic lives in those modules so that Phase 2+ notebooks/scripts can
import and reuse it without duplication or drift.

> **Note on data:** the real Indiana LendingClub extract
> (~37,515 rows, ~27.5 MB) should be placed at `data/raw/lendingclub_indiana_raw.csv`
> before running this notebook for the actual project. A small synthetic
> fixture generator (`tests/generate_synthetic_fixture.py`) is included
> only to verify the pipeline runs correctly in the absence of the real
> file — it is NOT a substitute for the genuine dataset.


In [1]:
# If the real raw data file is not yet present, uncomment the line below
# to generate a small synthetic fixture so the rest of this notebook runs
# end-to-end. Replace with the real LendingClub Indiana extract for the
# actual analysis.

# import subprocess, sys
# subprocess.run([sys.executable, "../tests/generate_synthetic_fixture.py"])


In [2]:
import sys
from pathlib import Path

# Allow imports from the project's src/ package when running from notebooks/
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np

from src import config, utils

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)


## 1. Project Setup
Ensure all project directories exist.

In [3]:
utils.ensure_directories()
print("Project root:", config.PROJECT_ROOT)
print("Raw data expected at:", config.RAW_DATA_PATH)


2026-07-29 00:14:24 | INFO     | src.utils | Verified 8 project directories exist.


Project root: /home/claude/mgmt590_capstone
Raw data expected at: /home/claude/mgmt590_capstone/data/raw/lendingclub_indiana_raw.csv


## 2. Data Ingestion
Load the raw CSV extract using the reusable `load_raw_data()` loader,
which raises clear, actionable errors if the file is missing, empty, or
unparsable.

In [4]:
raw_df = utils.load_raw_data()
print("Raw shape:", raw_df.shape)
raw_df.head()


2026-07-29 00:14:24 | INFO     | src.utils | Loaded raw dataset from /home/claude/mgmt590_capstone/data/raw/lendingclub_indiana_raw.csv — shape=(2015, 33)


Raw shape: (2015, 33)


,id,member_id,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,application_type,mort_acc,pub_rec_bankruptcies,desc,url
0,1,100000,26911,60 months,14.11%,803.94,D,D3,NaN,8 years,MORTGAGE,36118.90,Source Verified,Jun-2016,Charged Off,home_improvement,home_improvement,467xx,OH,9.27,0,Jan-1998,15,0,14340,26.4%,26,w,Individual,6,0,NaN,NaN
1,2,100001,33815,36 months,8.15%,1107.18,B,B1,NaN,10+ years,RENT,31051.48,Source Verified,Jan-2015,Charged Off,major_purchase,major_purchase,466xx,IN,9.02,1,Mar-2010,6,0,39271,33.3%,51,w,Individual,6,0,NaN,NaN
2,3,100002,7597,60 months,11.38%,126.32,E,E1,NaN,1 year,OWN,37962.44,Verified,Mar-2017,Fully Paid,credit_card,credit_card,462xx,IN,14.53,1,Jan-1998,2,0,59398,49.7%,23,w,Individual,6,0,NaN,NaN
3,4,100003,21693,36 months,10.45%,1184.56,C,C2,Nurse,2 years,RENT,75928.07,Verified,Mar-2017,Fully Paid,credit_card,credit_card,461xx,IN,15.44,0,Jun-2002,18,0,50455,72.8%,39,f,Individual,6,0,NaN,NaN
4,5,100004,37836,36 months,9.06%,446.26,A,A1,Manager,< 1 year,MORTGAGE,23044.16,Source Verified,Mar-2017,Fully Paid,other,other,473xx,IN,17.38,0,Mar-2010,14,0,8301,39.8%,50,w,Individual,0,0,NaN,NaN


In [5]:
raw_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 2015 entries, 0 to 2014
Data columns (total 33 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    2015 non-null   int64  
 1   member_id             2015 non-null   int64  
 2   loan_amnt             2015 non-null   int64  
 3   term                  2015 non-null   str    
 4   int_rate              2015 non-null   str    
 5   installment           2015 non-null   float64
 6   grade                 2015 non-null   str    
 7   sub_grade             2015 non-null   str    
 8   emp_title             1450 non-null   str    
 9   emp_length            1838 non-null   str    
 10  home_ownership        2015 non-null   str    
 11  annual_inc            1975 non-null   float64
 12  verification_status   2015 non-null   str    
 13  issue_d               2015 non-null   str    
 14  loan_status           2015 non-null   str    
 15  purpose               2015 non-n

## 3. Data Validation
`validate_dataset()` runs a structured battery of checks — schema drift,
missing values, duplicate rows, dtypes, and known business-rule
violations — and returns a report dict without mutating the data.

In [6]:
validation_report = utils.validate_dataset(raw_df)

print(f"Rows: {validation_report['n_rows']:,} | Columns: {validation_report['n_columns']}")
print(f"Duplicate rows: {validation_report['duplicate_row_count']:,}")
print(f"Missing expected columns: {validation_report['missing_columns']}")
print("\nColumns with missing values:")
validation_report["missing_values"]


2026-07-29 00:14:24 | INFO     | src.utils | Validation complete: 2015 rows, 33 columns, 15 duplicate rows, 5 columns with missing values, 0 missing expected columns.


Rows: 2,015 | Columns: 33
Duplicate rows: 15
Missing expected columns: []

Columns with missing values:


,missing_count,missing_pct
url,2015,100.00
desc,2015,100.00
emp_title,565,28.04
emp_length,177,8.78
annual_inc,40,1.99


In [7]:
print("Invalid / out-of-scope value checks:")
for check, count in validation_report["invalid_values"].items():
    print(f"  {check}: {count:,}")


Invalid / out-of-scope value checks:
  negative_or_zero_loan_amnt: 0
  negative_annual_inc: 0
  loan_status_out_of_scope_rows: 349
  rows_outside_target_state: 192


In [8]:
validation_report["dtypes"]


id                        int64
member_id                 int64
loan_amnt                 int64
term                        str
int_rate                    str
installment             float64
grade                       str
sub_grade                   str
emp_title                   str
emp_length                  str
home_ownership              str
annual_inc              float64
verification_status         str
issue_d                     str
loan_status                 str
purpose                     str
title                       str
zip_code                    str
addr_state                  str
dti                     float64
delinq_2yrs               int64
earliest_cr_line            str
open_acc                  int64
pub_rec                   int64
revol_bal                 int64
revol_util                  str
total_acc                 int64
initial_list_status         str
application_type            str
mort_acc                  int64
pub_rec_bankruptcies      int64
desc    

## 4. Data Cleaning & Feature Engineering
`clean_dataset()` orchestrates the full Phase 1 cleaning sequence:

1. Filter to Indiana borrowers (`addr_state == 'IN'`)
2. Remove exact duplicate rows
3. Convert percentage-string columns (`int_rate`, `revol_util`) to numeric
4. Parse `emp_length` free text into numeric years (`emp_length_years`)
5. Build the binary target `default_flag`
   (`Charged Off`/`Default` → 1, `Fully Paid` → 0; all other statuses —
   e.g. `Current`, `Late (31-120 days)` — are **removed** since those
   loans have not reached a final resolution)
6. Drop identifier / free-text / leakage-prone / superseded columns

Each step is also available individually in `src/utils.py` for
inspection or reuse.

In [9]:
cleaned_df = utils.clean_dataset(raw_df)
print("Cleaned shape:", cleaned_df.shape)
cleaned_df.head()


2026-07-29 00:14:24 | INFO     | src.utils | Beginning full cleaning pipeline on raw shape (2015, 33)


2026-07-29 00:14:24 | INFO     | src.utils | Filtered to state='IN': 2015 -> 1823 rows.


2026-07-29 00:14:24 | INFO     | src.utils | Removed 13 duplicate rows (1823 -> 1810).


2026-07-29 00:14:24 | INFO     | src.utils | Converted 'int_rate' to numeric (0 NaNs after conversion).


2026-07-29 00:14:24 | INFO     | src.utils | Converted 'revol_util' to numeric (0 NaNs after conversion).


...

2026-07-29 00:14:24 | INFO     | src.utils | Built target 'default_flag' from 'loan_status': kept 1500/1810 rows (310 dropped as non-final status). Class balance: {0: 0.743, 1: 0.257}


2026-07-29 00:14:24 | INFO     | src.utils | Dropped 13 excluded columns: ['id', 'member_id', 'emp_title', 'url', 'desc', 'title', 'zip_code', 'addr_state', 'issue_d', 'earliest_cr_line', 'sub_grade', 'loan_status', 'emp_length']


2026-07-29 00:14:24 | INFO     | src.utils | Cleaning pipeline complete. Final shape: (1500, 22)


Cleaned shape: (1500, 22)


,loan_amnt,term,int_rate,installment,grade,home_ownership,annual_inc,verification_status,purpose,dti,delinq_2yrs,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,application_type,mort_acc,pub_rec_bankruptcies,emp_length_years,default_flag
0,33815,36 months,8.15,1107.18,B,RENT,31051.48,Source Verified,major_purchase,9.02,1,6,0,39271,33.3,51,w,Individual,6,0,10.0,1
1,7597,60 months,11.38,126.32,E,OWN,37962.44,Verified,credit_card,14.53,1,2,0,59398,49.7,23,w,Individual,6,0,1.0,0
2,21693,36 months,10.45,1184.56,C,RENT,75928.07,Verified,credit_card,15.44,0,18,0,50455,72.8,39,f,Individual,6,0,2.0,0
3,37836,36 months,9.06,446.26,A,MORTGAGE,23044.16,Source Verified,other,17.38,0,14,0,8301,39.8,50,w,Individual,0,0,0.0,0
4,31764,60 months,13.14,1241.62,G,RENT,67139.39,Source Verified,credit_card,22.96,1,19,0,3910,44.5,25,f,Joint App,4,0,5.0,0


In [10]:
print("Target class balance (default_flag):")
cleaned_df[config.TARGET_COLUMN].value_counts(normalize=True).round(4)


Target class balance (default_flag):


default_flag
0    0.7427
1    0.2573
Name: proportion, dtype: float64

In [11]:
# Persist the cleaned dataset for downstream phases / reporting.
utils.save_dataframe(cleaned_df, config.CLEANED_DATA_PATH)


2026-07-29 00:14:24 | INFO     | src.utils | Saved dataframe with shape (1500, 22) to /home/claude/mgmt590_capstone/data/processed/lendingclub_indiana_cleaned.csv


## 5. Train / Validation / Test Split (Leakage-Safe)
`split_data()` carves out the **test** set first, then the
**validation** set from the remaining pool, stratifying on the binary
target at each step. No preprocessing statistics are ever computed using
validation or test rows — the preprocessing pipeline in the next section
is fit **only** on `X_train`.

In [12]:
X_train, X_val, X_test, y_train, y_val, y_test = utils.split_data(cleaned_df)

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")
print(f"Default rate — train: {y_train.mean():.3f} | val: {y_val.mean():.3f} | test: {y_test.mean():.3f}")


2026-07-29 00:14:24 | INFO     | src.utils | Split complete — train=1049 (69.9%), val=226 (15.1%), test=225 (15.0%). Default rate — train=0.257, val=0.257, test=0.258


Train: (1049, 21) | Val: (226, 21) | Test: (225, 21)
Default rate — train: 0.257 | val: 0.257 | test: 0.258


In [13]:
utils.save_splits(X_train, X_val, X_test, y_train, y_val, y_test)


2026-07-29 00:14:24 | INFO     | src.utils | Saved dataframe with shape (1049, 21) to /home/claude/mgmt590_capstone/data/splits/X_train.csv


2026-07-29 00:14:24 | INFO     | src.utils | Saved dataframe with shape (226, 21) to /home/claude/mgmt590_capstone/data/splits/X_val.csv


2026-07-29 00:14:24 | INFO     | src.utils | Saved dataframe with shape (225, 21) to /home/claude/mgmt590_capstone/data/splits/X_test.csv


2026-07-29 00:14:24 | INFO     | src.utils | Saved dataframe with shape (1049, 1) to /home/claude/mgmt590_capstone/data/splits/y_train.csv


2026-07-29 00:14:24 | INFO     | src.utils | Saved dataframe with shape (226, 1) to /home/claude/mgmt590_capstone/data/splits/y_val.csv


2026-07-29 00:14:24 | INFO     | src.utils | Saved dataframe with shape (225, 1) to /home/claude/mgmt590_capstone/data/splits/y_test.csv


2026-07-29 00:14:24 | INFO     | src.utils | All six train/validation/test split artifacts saved.


## 6. Preprocessing Pipeline (ColumnTransformer + Pipeline)
`build_preprocessing_pipeline()` returns an **unfitted**
`ColumnTransformer` with three branches:

- **Numeric** (`SimpleImputer(median)` → `StandardScaler`)
- **One-hot categorical** (`SimpleImputer(most_frequent)` → `OneHotEncoder`)
- **Ordinal categorical** — `grade`, encoded with its natural risk
  ordering A < B < ... < G (`SimpleImputer(most_frequent)` →
  `OrdinalEncoder`)

It is fit **only** on `X_train` to prevent data leakage, then reused
(`.transform()`) on validation/test/live data in later phases.

In [14]:
preprocessor = utils.build_preprocessing_pipeline()
preprocessor.fit(X_train)

feature_names = utils.get_output_feature_names(preprocessor)
print(f"Fitted. Output feature count: {len(feature_names)}")
feature_names[:15]


2026-07-29 00:14:24 | INFO     | src.utils | Built preprocessing ColumnTransformer: 14 numeric, 6 one-hot, 1 ordinal features.


Fitted. Output feature count: 34


['numeric__loan_amnt',
 'numeric__int_rate',
 'numeric__installment',
 'numeric__annual_inc',
 'numeric__dti',
 'numeric__delinq_2yrs',
 'numeric__open_acc',
 'numeric__pub_rec',
 'numeric__revol_bal',
 'numeric__revol_util',
 'numeric__total_acc',
 'numeric__mort_acc',
 'numeric__pub_rec_bankruptcies',
 'numeric__emp_length_years',
 'onehot_categorical__term_ 36 months']

In [15]:
X_train_transformed = preprocessor.transform(X_train)
print("Transformed training feature matrix shape:", X_train_transformed.shape)


Transformed training feature matrix shape: (1049, 34)


## 7. Serialize the Fitted Preprocessor
Saved via `joblib` for reuse in Phase 2 model training and the final Streamlit app.

In [16]:
utils.save_object(preprocessor, config.PREPROCESSOR_PATH)
print("Saved to:", config.PREPROCESSOR_PATH)


2026-07-29 00:14:25 | INFO     | src.utils | Serialized object to /home/claude/mgmt590_capstone/pipelines/preprocessing_pipeline.joblib


Saved to: /home/claude/mgmt590_capstone/pipelines/preprocessing_pipeline.joblib


## 8. Phase 1 Summary

| Artifact | Path |
|---|---|
| Cleaned dataset | `data/processed/lendingclub_indiana_cleaned.csv` |
| Train/val/test splits | `data/splits/*.csv` |
| Fitted preprocessing pipeline | `pipelines/preprocessing_pipeline.joblib` |
| Pipeline run log | `logs/pipeline.log` |

**Not implemented in Phase 1 (by design):** Logistic Regression, Random
Forest, and XGBoost model training and evaluation. These are stubbed out
with fixed signatures in `src/train_models.py`
(`train_logistic_regression`, `train_random_forest`, `train_xgboost`,
`evaluate_model`) and will be implemented in Phase 2 without requiring
any changes to the modules built here.

### Equivalent one-line pipeline run
Everything above can also be run non-interactively via:
```python
from src.train_models import run_phase1_pipeline
artifacts = run_phase1_pipeline()
```
or from the command line: `python -m src.train_models`
